# Pull images and annotations from Roboflow

**Author:** Juliana Thomasia C. Marcos\
**Input:** Roboflow project information (e.g., `rf_project_id`)\
**Output:** Images and annotation files stored in a Databricks volume

## Overview

This notebook downloads images and their corresponding annotations from an existing Roboflow project into a specified Databricks volume. It retrieves the latest available version of the annotation data using the provided Roboflow project ID and annotation type, making the dataset accessible for subsequent processing and integration into the Databricks MLOps workflows.

The downloaded data is organized within a structured directory based on the project and annotation type. Upon successful completion, the notebook provides a direct link to the destination folder in Databricks, allowing users to access and verify the retrieved data.

# Packages import and installation

In [0]:
%pip install ..

In [0]:
dbutils.library.restartPython()

In [0]:
from roboflow import Roboflow
from datetime import datetime as dt

from ag_vision.data_io import roboflow_io as rio
from ag_vision.constants import paths

# Variables set up

In [0]:
catalog_name = "use1_prod_artemis_catalog_3718194974443840"

In [0]:
api_key = dbutils.secrets.get("enablement-package", "roboflow_api_key")

In [0]:
rf_project_id = dbutils.widgets.get("rf_project_id")
annotation_type = dbutils.widgets.get("annotation_type")
databricks_folder_name = dbutils.widgets.get("databricks_folder_name")

In [0]:
databricks_project_path = f"/Volumes/{catalog_name}/production/data/{databricks_folder_name}"

In [0]:
rf_workspace = "cgiar-workspace"
rf = Roboflow(api_key=api_key)
rf_workspace = rf.workspace(rf_workspace)

# Annotations data download from Roboflow to Databricks

In [0]:
if rf_project_id:
    task_name=rf_project_id.replace('-', '_')
    try:
        rio.download_annotation_batch_from_roboflow(rf_workspace=rf_workspace,
                                                    rf_project_id=rf_project_id, 
                                                    project_path=databricks_project_path,
                                                    annotation_type=annotation_type,
                                                    task_name=task_name, 
                                                    download_date=dt.now().strftime("%m/%d/%Y"),
                                                    platform='db')
    except Exception as e:
        print(f"\n Something wrong happened. Please, check the information entered for the Roboflow project's id and the annotation type")
else:
    print("\n No Roboflow project's id provided. Please provide a relevant project id to download the data.")

In [0]:
annotation_download_path = paths.annotation_image_path(project=databricks_project_path,
                                                       annotation_type=annotation_type,
                                                       task_name=task_name,
                                                       batch_name='roboflow',
                                                       f_name='*.jpg')

if annotation_download_path:
    paths.show_volume_folder_link(path=annotation_download_path,
                                  url_name="Annotations folder",
                                  prefix=f"The latest version of the annotation data available in the Roboflow project with id {rf_project_id} is now available at: "
                            )